# LeetCode 430 — Flatten a Multilevel Doubly Linked List bằng Python

Link đề: https://leetcode.com/problems/flatten-a-multilevel-doubly-linked-list/ · Độ khó: **Medium** · Chủ đề: Linked List, Doubly-Linked List, Depth-First Search, Stack

Skeleton LeetCode:

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val, prev, next, child):
        self.val = val
        self.prev = prev
        self.next = next
        self.child = child
"""

class Solution:
    def flatten(self, head: 'Optional[Node]') -> 'Optional[Node]':
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án đã được kiểm chứng bằng code (kiểm tra con trỏ `prev/next/child` của kết quả, đối chiếu bốn cách trên hàng trăm danh sách đa tầng ngẫu nhiên). Cách LeetCode tuần tự hóa danh sách đa tầng ở dạng list phẳng có chút phức tạp; trong notebook mình dùng **biểu diễn lồng nhau tự dựng** (mục 3.2) để dựng dữ liệu — nếu cần khớp từng chữ với dữ liệu thật của LeetCode thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một danh sách liên kết **đôi** mà mỗi nút có thêm con trỏ **`child`** trỏ tới một danh sách đôi khác (và các danh sách con lại có thể có con...). Phải **làm phẳng** thành **một** danh sách đôi: các nút của danh sách con xuất hiện **ngay sau nút cha** và **trước `cha.next`**; mọi `child` phải thành `None`.
2. Nắm các công cụ: lớp `Node` với `prev/next/child`, kỹ thuật **chèn nguyên một danh sách vào giữa hai nút** (thứ tự gán con trỏ rất quan trọng), **ngăn xếp** để "nhớ phần còn lại", đệ quy **trả về nút cuối (tail)**, và hàm kiểm tra tính hợp lệ cho danh sách đôi.
3. Đi từ **gom nút theo tiền thứ tự rồi nối lại** → **DFS đệ quy trả về tail** → **ngăn xếp** → **"nối ghép tại chỗ" `O(1)` bộ nhớ**.
4. Hiểu mấu chốt: làm phẳng chính là **duyệt tiền thứ tự (preorder)** của cây ẩn: thăm nút, rồi **toàn bộ `child` trước**, rồi mới tới `next`. Bài tập thực chất là *thăm đúng thứ tự* và *nối lại `prev/next` cho đúng*.
5. Tránh các lỗi kinh điển: quên đặt `child = None`, quên cập nhật **`prev`** (hay làm mỗi `next`), làm mất `cur.next` khi ghi đè, thứ tự đẩy ngăn xếp sai, và quên danh sách rỗng.

## 2. Đề bài & ví dụ

Cho `head` của tầng đầu tiên của một danh sách đôi; mỗi nút có `next`, `prev` và thêm `child` (có thể trỏ tới danh sách đôi khác, cũng có nút với `child` riêng...). Hãy **làm phẳng** để mọi nút nằm trong **một danh sách đôi một tầng**. Với nút `curr` có danh sách con, các nút của danh sách con phải xuất hiện **sau `curr`** và **trước `curr.next`**. Trả về `head` của danh sách phẳng; mọi `child` phải là `None`.

**Ví dụ 1** — cấu trúc:

```
1---2---3---4---5---6--NULL
        |
        7---8---9---10--NULL
            |
            11--12--NULL
```
Kết quả: `[1,2,3,7,8,11,12,9,10,4,5,6]`. Nút `3` có con `7`; nút `8` có con `11`. Sau `3` đến cả danh sách `7,8,...`; và sau `8` đến cả `11,12` rồi mới `9,10`, cuối cùng `4,5,6`.

**Ví dụ 2** — đề gốc cho `head = [1,2,null,3]`: tầng chính `1---2`, và nút `1` có con `3` (danh sách `3---NULL`). Kết quả `[1,3,2]`: con `3` chèn ngay sau `1`, trước `2`.

**Ví dụ 3** — `head = []` → `[]`.

**Ràng buộc:** số nút không quá `1000`; `1 <= Node.val <= 10^5`.

**Lưu ý đối chiếu:** hình minh họa Ví dụ 2 của đề quyết định nút nào có con; mình hiểu là nút `1` có `child = 3` (cho ra đúng `[1,3,2]`) — nếu cần hãy **đối chiếu lại đề gốc**.

In [1]:
# Ví dụ tự dựng theo mục 3.2 (spec lồng nhau); ở đây chỉ in cấu trúc mô tả
spec_ex1 = [1, 2, (3, [7, (8, [11, 12]), 9, 10]), 4, 5, 6]
spec_ex2 = [(1, [3]), 2]
spec_ex3 = []
print(spec_ex1)
print(spec_ex2)

[1, 2, (3, [7, (8, [11, 12]), 9, 10]), 4, 5, 6]
[(1, [3]), 2]


## 3. Kiến thức nền

### 3.1. Nút `Node` — ba con trỏ

Nút của danh sách đôi thường chỉ có `prev`, `next`. Ở đây có thêm **`child`**: trỏ tới **đầu** của một danh sách đôi khác ("tầng dưới"). Hãy hình dung mỗi nút là một **cây** mà `next` là "anh em kế tiếp" còn `child` là "con".

In [2]:
class Node:
    def __init__(self, val, prev=None, next=None, child=None):
        self.val = val
        self.prev = prev
        self.next = next
        self.child = child

    def __repr__(self):
        return f"Node({self.val})"


a, b = Node(1), Node(2)
a.next, b.prev = b, a            # hai chiều: a <-> b
c = Node(3)
a.child = c                      # tầng dưới của a: danh sách một nút [3]
print(a.next, b.prev, a.child, a.prev)

Node(2) Node(1) Node(3) None


### 3.2. Dựng dữ liệu từ biểu diễn lồng nhau `spec`

LeetCode tuần tự hóa danh sách đa tầng bằng một list phẳng có nhiều `null` (xem đề). Để dựng dữ liệu cho dễ, mình dùng **biểu diễn lồng nhau (tự dựng)**: một **list**; mỗi phần tử là

- số nguyên `v`: nút `v` **không** có con; hoặc
- tuple `(v, sub)`: nút `v` có con là danh sách `sub` (cũng cùng quy cách).

Ví dụ 1 của đề viết thành: `[1, 2, (3, [7, (8, [11, 12]), 9, 10]), 4, 5, 6]`. Hàm `build_multilevel(spec)` nối `next/prev` trong từng tầng và gán `child`. Hàm `flat_spec(spec)` là **đáp án mong đợi** tính **trực tiếp từ spec** (không đụng con trỏ): với mỗi phần tử xuất ra `v`, rồi nếu có con thì xuất toàn bộ con (đệ quy), rồi mới sang phần tử kế.

In [3]:
def build_multilevel(spec):
    """Dựng danh sách đa tầng từ spec lồng nhau; trả về head (None nếu rỗng)."""
    head = prev = None
    for item in spec:
        if isinstance(item, tuple):
            val, sub = item
        else:
            val, sub = item, []
        node = Node(val)
        node.child = build_multilevel(sub) if sub else None
        if prev is None:
            head = node
        else:
            prev.next = node
            node.prev = prev
        prev = node
    return head


def flat_spec(spec):
    """Đáp án mong đợi tính thẳng từ spec (độc lập với con trỏ)."""
    out = []
    for item in spec:
        if isinstance(item, tuple):
            val, sub = item
            out.append(val)
            out.extend(flat_spec(sub))
        else:
            out.append(item)
    return out


h1 = build_multilevel(spec_ex1)
print(h1.val, h1.next.val, h1.next.next.val, "| child của 3:", h1.next.next.child.val)
print(flat_spec(spec_ex1))          # [1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6]
print(flat_spec(spec_ex2))          # [1, 3, 2]

1 2 3 | child của 3: 7
[1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6]
[1, 3, 2]


### 3.3. In cấu trúc đa tầng để "nhìn thấy"

Hàm `show_multilevel` in mỗi tầng thụt vào một mức: mỗi nút một dòng, kèm `child` nếu có. Bạn nên chạy nó **trước và sau** khi làm phẳng để thấy sự khác biệt.

In [4]:
def show_multilevel(head, depth=0):
    cur = head
    while cur is not None:
        print("    " * depth + str(cur.val))
        if cur.child is not None:
            show_multilevel(cur.child, depth + 1)
        cur = cur.next

show_multilevel(build_multilevel(spec_ex1))

1
2
3
    7
    8
        11
        12
    9
    10
4
5
6


### 3.4. Chèn nguyên một danh sách vào giữa hai nút (thứ tự gán rất quan trọng)

Cho nút `cur`, danh sách con bắt đầu `child` và kết thúc `tail`; và `nxt = cur.next` (có thể `None`). Muốn: `cur → child ... tail → nxt`, và chiều ngược lại cho `prev`. Bốn việc:

1. **Lưu `nxt = cur.next` trước!** (nếu ghi đè `cur.next` trước thì mất phần còn lại)
2. `cur.next = child` và `child.prev = cur`.
3. `tail.next = nxt`.
4. Nếu `nxt` không `None`: `nxt.prev = tail`.

Sau đó `cur.child = None`.

In [5]:
x = build_multilevel([1, (2, [10, 11]), 3])
cur = x.next                       # nút 2, có con 10-11
child = cur.child
tail = child
while tail.next:
    tail = tail.next               # tail = nút 11
nxt = cur.next                     # LƯU TRƯỚC: nút 3

cur.next, child.prev = child, cur
tail.next = nxt
if nxt:
    nxt.prev = tail
cur.child = None

node, vals = x, []
while node:
    vals.append(node.val); node = node.next
print(vals)                        # [1, 2, 10, 11, 3]
print(nxt.prev, tail.next)         # Node(11) Node(3)

[1, 2, 10, 11, 3]
Node(11) Node(3)


### 3.5. Thứ tự làm phẳng = tiền thứ tự (preorder)

Nếu coi `child` là "con" và `next` là "em kế", thì thứ tự yêu cầu là: **thăm nút → thăm hết `child` (cả dòng con cháu) → rồi tới `next`**. Đó là **tiền thứ tự** của một cây mà mỗi nút có (tối đa) hai nhánh: `child` được đi trước `next`. Ví dụ 1: `1, 2, 3, [7, 8, [11, 12], 9, 10], 4, 5, 6`.

### 3.6. Ngăn xếp (stack) để nhớ "phần còn lại"

Khi rẽ xuống `child`, ta phải nhớ `next` của nút hiện tại để quay lại sau. Ngăn xếp (`list` với `append`/`pop`, LIFO) làm đúng việc đó: **đẩy `next` trước, rồi đẩy `child`** thì `pop` ra `child` trước (đúng yêu cầu), còn `next` đợi dưới đáy.

In [6]:
stack = []
stack.append("next của 3 (tức 4)")
stack.append("child của 3 (tức 7)")
print(stack.pop())     # child trước
print(stack.pop())     # rồi mới tới next

child của 3 (tức 7)
next của 3 (tức 4)


### 3.7. Giám khảo `check_flat`

Bài trả về **cấu trúc con trỏ**, nên cần một giám khảo đi cả hai chiều. `check_flat(head, expected, original_ids)` kiểm tra:

1. Rỗng ⇔ `expected == []` và `head is None`.
2. Đi `next` từ `head`: giá trị khớp `expected`; **mọi `child` là `None`**; `head.prev is None`; với mọi nút `x.next`, ta có `x.next.prev is x`; cuối cùng `next` là `None`; có bảo vệ chống vòng lặp.
3. Đi ngược `prev` từ nút cuối ra dãy đảo.
4. Tập đối tượng nút (theo `id`) **không đổi** — ta chỉ nối lại chứ không tạo nút mới.

Hàm `all_nodes(head)` gom mọi nút của cấu trúc **trước** khi làm phẳng (qua `next` và `child`, dùng ngăn xếp).

In [7]:
def all_nodes(head):
    out, stack = [], [head] if head else []
    while stack:
        cur = stack.pop()
        while cur is not None:
            out.append(cur)
            if cur.child is not None:
                stack.append(cur.child)
            cur = cur.next
    return out


def check_flat(head, expected, original_ids):
    if not expected:
        return (head is None, "rỗng" if head is None else "đáng lẽ None")
    if head is None:
        return False, "head là None"
    if head.prev is not None:
        return False, "head.prev phải là None"
    vals, ids, cur, last = [], set(), head, None
    while cur is not None:
        if id(cur) in ids or len(vals) > len(expected) + 5:
            return False, "có vòng lặp / dài quá mức"
        ids.add(id(cur)); vals.append(cur.val)
        if cur.child is not None:
            return False, f"nút {cur.val} vẫn còn child"
        if cur.next is not None and cur.next.prev is not cur:
            return False, f"prev/next lệch tại nút {cur.val}"
        last, cur = cur, cur.next
    if vals != expected:
        return False, f"thứ tự sai: {vals}"
    back, cur = [], last
    while cur is not None and len(back) <= len(expected):
        back.append(cur.val); cur = cur.prev
    if back != expected[::-1]:
        return False, f"đi ngược sai: {back}"
    if ids != original_ids:
        return False, "tập nút thay đổi (tạo/mất nút)"
    return True, "ok"


h = build_multilevel(spec_ex1)
print(check_flat(h, flat_spec(spec_ex1), {id(n) for n in all_nodes(h)}))   # còn child => False

(False, 'nút 3 vẫn còn child')


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: tầng chính `1 2 3 4 5 6`; nút `3` có con `[7 8 9 10]`; nút `8` có con `[11 12]`.

Đi từ trái sang phải, mỗi lần gặp nút có `child` thì **rẽ xuống ngay**, đi hết danh sách con (và con cháu của nó) rồi mới quay lại tiếp phần sau của nút cha:

| Bước | Đang ở | Hành động | Dãy kết quả |
|---|---|---|---|
| 1 | `1` | không con → viết `1` | `1` |
| 2 | `2` | không con → viết `2` | `1 2` |
| 3 | `3` | viết `3`, **có con `7`** → rẽ xuống; nhớ "sau này quay về `4`" | `1 2 3` |
| 4 | `7` | viết `7` | `... 7` |
| 5 | `8` | viết `8`, **có con `11`** → rẽ; nhớ "quay về `9`" | `... 7 8` |
| 6 | `11, 12` | viết `11`, `12`; hết danh sách → quay về `9` | `... 11 12` |
| 7 | `9, 10` | viết `9`, `10`; hết → quay về `4` | `... 9 10` |
| 8 | `4, 5, 6` | viết hết | `1 2 3 7 8 11 12 9 10 4 5 6` |

Cái cần "nhớ" khi rẽ xuống là **`next` của nút đang đứng** — chính là việc của ngăn xếp (hay của lời gọi đệ quy). Phần việc còn lại là **nối đúng `prev/next`** và **xóa `child`**.

## 5. Cách 1 — Brute Force / baseline: gom các nút theo tiền thứ tự rồi nối lại

**Ý tưởng.** Tách làm hai bước dễ hiểu:

1. DFS tiền thứ tự để **gom mọi nút (đối tượng) vào một list** `nodes`: với mỗi nút ghi nó, rồi đi vào `child`, rồi sang `next`. Ghi lại các nút **trước khi** sửa con trỏ.
2. Duyệt `nodes`: với mỗi `i`, `nodes[i].prev = nodes[i-1]` (hoặc `None` nếu `i = 0`), `nodes[i].next = nodes[i+1]` (hoặc `None` nếu cuối), và `child = None`.

In [8]:
def flatten_collect(head):
    if head is None:
        return None
    nodes = []

    def preorder(node):
        while node is not None:
            nodes.append(node)
            if node.child is not None:
                preorder(node.child)
            node = node.next

    preorder(head)
    for i, node in enumerate(nodes):
        node.prev = nodes[i - 1] if i > 0 else None
        node.next = nodes[i + 1] if i + 1 < len(nodes) else None
        node.child = None
    return nodes[0]


def read_vals(head):
    out = []
    while head is not None and len(out) < 5000:
        out.append(head.val); head = head.next
    return out

for spec in (spec_ex1, spec_ex2, spec_ex3):
    print(read_vals(flatten_collect(build_multilevel(spec))), "(kỳ vọng", flat_spec(spec), ")")

[1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6] (kỳ vọng [1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6] )
[1, 3, 2] (kỳ vọng [1, 3, 2] )
[] (kỳ vọng [] )


**Độ phức tạp (Cách 1).** Time **`O(n)`** (mỗi nút ghi một lần, nối lại một lần). Space **`O(n)`** cho `nodes` (cộng `O(độ sâu)` ngăn xếp đệ quy).

**Nút thắt:** ta lưu cả `n` nút chỉ để sửa lại con trỏ. Thực ra chỉ cần nhớ "nút đứng trước" và các `next` đang chờ.

## 6. Ý tưởng thuật toán / bước đột phá

Hai điểm mấu chốt:

1. **Nối từng đoạn tại chỗ.** Khi gặp nút `cur` có `child`, ta **chèn nguyên danh sách con** vào giữa `cur` và `cur.next` (4 phép gán ở mục 3.4). Sau khi chèn, danh sách con **đã nằm trong tầng chính**; khi `cur` tiến tới (`cur = cur.next` = `child`), ta sẽ tự động gặp các nút của danh sách con và xử lý `child` của chúng sau — **không cần đệ quy hay ngăn xếp** (Cách 4).
2. Nếu thích tư duy "đi xuống rồi quay lại": hàm đệ quy `dfs(node)` làm phẳng danh sách bắt đầu từ `node` và **trả về nút cuối (tail)** của nó, để cha biết nối `tail.next = cha.next` (Cách 2). Hoặc dùng ngăn xếp nhớ `next` (Cách 3).

Cả bốn cách đều `O(n)` thời gian; khác ở bộ nhớ phụ: Cách 4 chỉ `O(1)`.

## 7. Các cách tiếp theo

### Cách 2 — DFS đệ quy, trả về nút cuối (tail)

**Giải thích từng đoạn.** `dfs(node)` làm phẳng **danh sách bắt đầu ở `node`** (cùng mọi con cháu) và **trả về nút cuối cùng** của danh sách phẳng đó.

1. Duyệt `cur` từ `node` qua `next`. Với mỗi `cur`, **lưu `nxt = cur.next`** trước.
2. Nếu `cur.child`: `child_tail = dfs(cur.child)` (con đã phẳng, có tail); chèn: `cur.next = cur.child; cur.child.prev = cur; child_tail.next = nxt; nếu nxt: nxt.prev = child_tail; cur.child = None`; `tail = child_tail`.
3. Ngược lại `tail = cur`.
4. `cur = nxt` (nhảy sang nút kế **gốc**, sau đoạn vừa chèn). Cuối vòng trả `tail`.

In [9]:
def flatten_dfs(head):
    def dfs(node):
        """Làm phẳng danh sách bắt đầu ở node; trả về nút cuối của nó."""
        cur, tail = node, node
        while cur is not None:
            nxt = cur.next                          # LƯU TRƯỚC khi nối
            if cur.child is not None:
                child_tail = dfs(cur.child)         # con đã phẳng, biết tail
                cur.next = cur.child
                cur.child.prev = cur
                child_tail.next = nxt
                if nxt is not None:
                    nxt.prev = child_tail
                cur.child = None
                tail = child_tail
            else:
                tail = cur
            cur = nxt
        return tail

    if head is not None:
        dfs(head)
    return head


for spec in (spec_ex1, spec_ex2, spec_ex3):
    print(read_vals(flatten_dfs(build_multilevel(spec))), "(kỳ vọng", flat_spec(spec), ")")

[1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6] (kỳ vọng [1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6] )
[1, 3, 2] (kỳ vọng [1, 3, 2] )
[] (kỳ vọng [] )


In [10]:
class Solution2:
    def flatten(self, head: 'Optional[Node]') -> 'Optional[Node]':
        def dfs(node):
            cur, tail = node, node
            while cur is not None:
                nxt = cur.next
                if cur.child is not None:
                    child_tail = dfs(cur.child)
                    cur.next = cur.child
                    cur.child.prev = cur
                    child_tail.next = nxt
                    if nxt is not None:
                        nxt.prev = child_tail
                    cur.child = None
                    tail = child_tail
                else:
                    tail = cur
                cur = nxt
            return tail

        if head is not None:
            dfs(head)
        return head


h = Solution2().flatten(build_multilevel(spec_ex1))
print(read_vals(h))

[1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6]


**Độ phức tạp (Cách 2).** Mỗi nút được thăm đúng một lần ⇒ Time **`O(n)`**. Space **`O(d)`** ngăn xếp đệ quy, `d` là **độ sâu lồng nhau** lớn nhất (tệ nhất `O(n)` — mỗi nút có con kế tiếp tạo chuỗi 1000 tầng, có thể chạm giới hạn đệ quy, xem mục 12.4).

### Cách 3 — Ngăn xếp (lặp), nối dần từng nút vào danh sách kết quả

**Giải thích từng đoạn.**

1. `dummy` là nút giả đứng trước đầu (tránh xử lý riêng cho nút đầu); `prev = dummy`.
2. `stack = [head]`. Lặp: `cur = stack.pop()`; **nối `prev ⇄ cur`** (`prev.next = cur; cur.prev = prev`).
3. **Đẩy `cur.next` trước, rồi đẩy `cur.child`** (nếu có), và đặt `cur.child = None`. (`child` pop ra trước, đúng yêu cầu.)
4. `prev = cur`.
5. Cuối: `head = dummy.next; head.prev = None`.

Ở đây ta **xây lại** danh sách theo đúng thứ tự `pop`, nên không cần "chèn giữa" như Cách 2/4.

In [11]:
def flatten_stack(head):
    if head is None:
        return None
    dummy = Node(0)
    prev = dummy
    stack = [head]
    while stack:
        cur = stack.pop()
        prev.next = cur
        cur.prev = prev
        if cur.next is not None:
            stack.append(cur.next)          # đẩy next TRƯỚC...
        if cur.child is not None:
            stack.append(cur.child)         # ...child SAU => child được pop trước
            cur.child = None
        prev = cur
    head = dummy.next
    head.prev = None
    return head


for spec in (spec_ex1, spec_ex2, spec_ex3):
    print(read_vals(flatten_stack(build_multilevel(spec))), "(kỳ vọng", flat_spec(spec), ")")

[1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6] (kỳ vọng [1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6] )
[1, 3, 2] (kỳ vọng [1, 3, 2] )
[] (kỳ vọng [] )


In [12]:
class Solution3:
    def flatten(self, head: 'Optional[Node]') -> 'Optional[Node]':
        if head is None:
            return None
        dummy = Node(0, None, head, None)
        prev = dummy
        stack = [head]
        while stack:
            cur = stack.pop()
            prev.next = cur
            cur.prev = prev
            if cur.next:
                stack.append(cur.next)
            if cur.child:
                stack.append(cur.child)
                cur.child = None
            prev = cur
        dummy.next.prev = None
        return dummy.next


print(read_vals(Solution3().flatten(build_multilevel(spec_ex1))))

[1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6]


**Độ phức tạp (Cách 3).** Time **`O(n)`**. Space **`O(d + w)`** — ngăn xếp chứa các `next` đang chờ, tệ nhất `O(n)`. Không bị giới hạn đệ quy.

### Cách 4 — "Nối ghép" tại chỗ, `O(1)` bộ nhớ phụ

Đây là cách khéo nhất: **không** đệ quy, **không** ngăn xếp. Với một con trỏ `cur` đi dọc tầng chính:

1. Nếu `cur.child` là `None`: `cur = cur.next`.
2. Nếu có `child`: tìm **tail** của danh sách con (đi `next` đến cuối). Lưu `nxt = cur.next`. Chèn bằng 4 phép gán (mục 3.4), `cur.child = None`.
3. Rồi `cur = cur.next` (lúc này chính là `child`), tiếp tục. Các nút của danh sách con **đã nằm trên tầng chính**, nên nếu chúng có `child`, ta sẽ xử lý ở các bước sau theo cùng quy tắc.

**Vì sao đúng thứ tự?** Danh sách con được chèn **ngay sau `cur`**, nên ta đi qua nó **trước** `nxt` — đúng "con trước, anh em sau", và con cháu của con cũng được chèn ngay sau chính nút của chúng, vẫn trước phần còn lại.

In [13]:
def flatten_splice(head):
    cur = head
    while cur is not None:
        if cur.child is not None:
            child = cur.child
            tail = child
            while tail.next is not None:        # tìm nút cuối của danh sách con
                tail = tail.next
            nxt = cur.next                      # LƯU TRƯỚC
            cur.next = child
            child.prev = cur
            tail.next = nxt
            if nxt is not None:
                nxt.prev = tail
            cur.child = None
        cur = cur.next
    return head


for spec in (spec_ex1, spec_ex2, spec_ex3):
    print(read_vals(flatten_splice(build_multilevel(spec))), "(kỳ vọng", flat_spec(spec), ")")

[1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6] (kỳ vọng [1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6] )
[1, 3, 2] (kỳ vọng [1, 3, 2] )
[] (kỳ vọng [] )


In [14]:
class Solution4:
    def flatten(self, head: 'Optional[Node]') -> 'Optional[Node]':
        cur = head
        while cur:
            if cur.child:
                child = cur.child
                tail = child
                while tail.next:
                    tail = tail.next
                nxt = cur.next
                cur.next = child
                child.prev = cur
                tail.next = nxt
                if nxt:
                    nxt.prev = tail
                cur.child = None
            cur = cur.next
        return head


print(read_vals(Solution4().flatten(build_multilevel(spec_ex1))))

[1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6]


**Độ phức tạp (Cách 4).** Time **`O(n)`**: việc đi tìm `tail` đi qua các nút **của chính danh sách con đó** (một nút chỉ bị đi qua bởi **đúng một** lần tìm tail — lần chèn danh sách chứa nó, vì lúc ấy các `child` của nó chưa được nối vào `next`), cộng lượt `cur` đi qua mỗi nút một lần. Space **`O(1)`**.

## 8. TRACE từng bước

Trace **Cách 4** (nối ghép) trên Ví dụ 1: in nút `cur`, danh sách con tìm được, tail, và dãy phẳng hiện tại (đọc theo `next` từ đầu) sau mỗi lần chèn.

In [15]:
def trace_splice(head):
    cur = head
    step = 0
    while cur is not None:
        if cur.child is not None:
            step += 1
            child = cur.child
            tail = child
            chain = [child.val]
            while tail.next is not None:
                tail = tail.next
                chain.append(tail.val)
            nxt = cur.next
            print(f"bước {step}: cur={cur.val} có child, danh sách con = {chain} (tail={tail.val}), nxt={nxt.val if nxt else None}")
            cur.next = child
            child.prev = cur
            tail.next = nxt
            if nxt is not None:
                nxt.prev = tail
            cur.child = None
            print(f"         sau khi chèn: tầng chính = {read_vals(head)}")
        cur = cur.next
    return head


h = trace_splice(build_multilevel(spec_ex1))
print("kết quả:", read_vals(h))
ok = check_flat(h, flat_spec(spec_ex1), {id(n) for n in all_nodes(build_multilevel(spec_ex1))})
print("(ids khác vì dựng lại cây mới để so — chỉ cần thấy thứ tự đúng)")
assert read_vals(h) == [1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6]

bước 1: cur=3 có child, danh sách con = [7, 8, 9, 10] (tail=10), nxt=4
         sau khi chèn: tầng chính = [1, 2, 3, 7, 8, 9, 10, 4, 5, 6]
bước 2: cur=8 có child, danh sách con = [11, 12] (tail=12), nxt=9
         sau khi chèn: tầng chính = [1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6]
kết quả: [1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6]
(ids khác vì dựng lại cây mới để so — chỉ cần thấy thứ tự đúng)


Bảng trạng thái (Cách 4, Ví dụ 1):

| Bước | `cur` | `child` tìm thấy | `tail` | `nxt` | Tầng chính sau khi chèn |
|---|---|---|---|---|---|
| 1 | `3` | `7 8 9 10` | `10` | `4` | `1 2 3 7 8 9 10 4 5 6` |
| 2 | `8` | `11 12` | `12` | `9` | `1 2 3 7 8 11 12 9 10 4 5 6` |

Chú ý: ở bước 1, nút `8` (có con `11 12`) **đã nằm trên tầng chính**; `cur` đi tiếp `7 → 8` thì mới bước 2. Khi chèn `11 12` vào giữa `8` và `9`, ta có đúng `... 8 11 12 9 ...`. Các nút `1, 2, 4, 5, 6, 7, 9, 10, 11, 12` không có `child` nên `cur` chỉ đi qua.

Trace thêm **Cách 3 (ngăn xếp)**: in `stack` và nút vừa `pop`.

In [16]:
def trace_stack(head):
    dummy = Node(0)
    prev = dummy
    stack = [head]
    while stack:
        cur = stack.pop()
        prev.next = cur
        cur.prev = prev
        pushed = []
        if cur.next is not None:
            stack.append(cur.next); pushed.append(f"next={cur.next.val}")
        if cur.child is not None:
            stack.append(cur.child); pushed.append(f"child={cur.child.val}")
            cur.child = None
        print(f"pop {cur.val:>2}: đẩy [{', '.join(pushed)}] -> stack = {[n.val for n in stack]}")
        prev = cur
    head = dummy.next
    head.prev = None
    return head


h = trace_stack(build_multilevel(spec_ex1))
print(read_vals(h))

pop  1: đẩy [next=2] -> stack = [2]
pop  2: đẩy [next=3] -> stack = [3]
pop  3: đẩy [next=4, child=7] -> stack = [4, 7]
pop  7: đẩy [next=8] -> stack = [4, 8]
pop  8: đẩy [next=9, child=11] -> stack = [4, 9, 11]
pop 11: đẩy [next=12] -> stack = [4, 9, 12]
pop 12: đẩy [] -> stack = [4, 9]
pop  9: đẩy [next=10] -> stack = [4, 10]
pop 10: đẩy [] -> stack = [4]
pop  4: đẩy [next=5] -> stack = [5]
pop  5: đẩy [next=6] -> stack = [6]
pop  6: đẩy [] -> stack = []
[1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6]


## 9. Code nộp LeetCode

Bản nộp (Cách 4: nối ghép, `O(1)` bộ nhớ phụ). Trên LeetCode **không** khai báo lại `Node`.

In [17]:
class Solution:
    def flatten(self, head: 'Optional[Node]') -> 'Optional[Node]':
        cur = head
        while cur:
            if cur.child:
                child = cur.child
                tail = child
                while tail.next:
                    tail = tail.next
                nxt = cur.next
                cur.next = child
                child.prev = cur
                tail.next = nxt
                if nxt:
                    nxt.prev = tail
                cur.child = None
            cur = cur.next
        return head


for spec in (spec_ex1, spec_ex2, spec_ex3):
    head = build_multilevel(spec)
    ids = {id(n) for n in all_nodes(head)}
    res = Solution().flatten(head)
    ok, msg = check_flat(res, flat_spec(spec), ids)
    print(read_vals(res), ok, msg)
    assert ok

[1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6] True ok
[1, 3, 2] True ok
[] True rỗng


## 10. Vì sao thuật toán đúng

**Cách 4 (nối ghép).** Bất biến: *khi `cur` đứng ở một nút, mọi nút **từ đầu đến trước `cur`** trên tầng chính đã được làm phẳng hoàn toàn (không còn `child` ở phía trước), và dãy từ `cur` trở đi, đọc theo `next`, gồm các nút còn lại **theo đúng thứ tự phẳng chỉ khác là các `child` chưa được nối**.* Khi `cur` có `child`, ta chèn danh sách con ngay sau `cur`: dãy phía sau `cur` thành `child..tail, nxt..`, đúng là "con trước, anh em sau". Danh sách con tự có thể có `child` ở các nút bên trong, nhưng những nút đó nằm sau `cur` ⇒ `cur` sẽ đi qua và xử lý, vẫn giữ bất biến. Mỗi lần chèn xóa một `child` khác `None`, nên cuối cùng không còn `child`. Con trỏ `prev` được cập nhật ở cả hai đầu mối nối (`child.prev = cur`, `nxt.prev = tail`), nên tính nhất quán hai chiều giữ nguyên.

**Cách 2.** Quy nạp theo độ sâu lồng nhau: `dfs(node)` trả danh sách phẳng + tail đúng cho danh sách con; ghép con vào giữa `cur` và `nxt` bằng 4 phép gán cho kết quả đúng.

**Cách 3.** Thứ tự `pop` là tiền thứ tự (con trước, `next` sau): vì mỗi nút `cur` đẩy `next` rồi mới đẩy `child`, nút cuối cùng của toàn bộ "nhánh con" được `pop` trước khi `next` của `cur` được `pop`. Ta nối `prev ⇄ cur` theo thứ tự `pop`, nên danh sách kết quả đúng thứ tự và hai chiều.

**Cách 1** chỉ là cùng thứ tự tiền thứ tự, tách thành "gom" rồi "nối".

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space phụ |
|---|---|---|
| 1. Gom nút rồi nối lại | `O(n)` | `O(n)` |
| 2. DFS đệ quy trả tail | `O(n)` | `O(d)` (độ sâu lồng nhau) |
| 3. Ngăn xếp | `O(n)` | `O(n)` tệ nhất |
| 4. Nối ghép tại chỗ | `O(n)` | **`O(1)`** |

`n` ≤ 1000 là số nút, `d` là độ sâu lồng nhau lớn nhất. Cả bốn cách đều rất nhanh với `n ≤ 1000`; lý do chọn Cách 4 là **không dính giới hạn đệ quy** và **bộ nhớ hằng số**.

## 12. Bộ test

Mỗi lần chạy một cách ta **dựng danh sách mới** từ `spec` (vì cách giải sửa tại chỗ). Đáp án mong đợi là `flat_spec(spec)` — tính **trực tiếp từ spec**, hoàn toàn độc lập với con trỏ.

In [18]:
import random, sys

ALL = {
    "cach1": flatten_collect,
    "cach2": flatten_dfs,
    "cach3": flatten_stack,
    "cach4": flatten_splice,
    "nop":   lambda h: Solution().flatten(h),
}

def run_all(spec):
    expected = flat_spec(spec)
    for name, fn in ALL.items():
        head = build_multilevel(spec)
        ids = {id(n) for n in all_nodes(head)}
        res = fn(head)
        ok, msg = check_flat(res, expected, ids)
        assert ok, (name, spec, msg)
    return expected

for sp in (spec_ex1, spec_ex2, spec_ex3):
    run_all(sp)
print("ví dụ của đề: OK")

ví dụ của đề: OK


### 12.1. Các trường hợp biên

In [19]:
print(run_all([]))                                   # rỗng
print(run_all([5]))                                  # một nút
print(run_all([1, 2, 3]))                            # không có child nào
print(run_all([(1, [2])]))                           # nút duy nhất có con duy nhất
print(run_all([(1, [(2, [(3, [4])])])]))             # chuỗi con lồng nhau
print(run_all([(1, [2, 3]), (4, [5, 6])]))           # nhiều nút có con ở cùng tầng
print(run_all([1, (2, [3, 4]), 5]))                  # child ở giữa
print(run_all([1, 2, (3, [4, 5])]))                  # child ở nút CUỐI (nxt = None)
print(run_all([(1, [(2, [3]), (4, [5])]), 6]))       # con có con ở nhiều nút
print("OK")

[]
[5]
[1, 2, 3]
[1, 2]
[1, 2, 3, 4]
[1, 2, 3, 4, 5, 6]
[1, 2, 3, 4, 5]
[1, 2, 3, 4, 5]
[1, 2, 3, 4, 5, 6]
OK


### 12.2. Phản ví dụ cho các cách làm sai

Các hàm sai điển hình (giải thích ở mục 13); giám khảo phải bắt được:

In [20]:
def wrong_keep_child(head):                 # quên đặt child = None
    cur = head
    while cur:
        if cur.child:
            child, tail = cur.child, cur.child
            while tail.next: tail = tail.next
            nxt = cur.next
            cur.next = child; child.prev = cur
            tail.next = nxt
            if nxt: nxt.prev = tail
            # THIẾU: cur.child = None
        cur = cur.next
    return head

def wrong_no_prev(head):                    # quên cập nhật prev
    cur = head
    while cur:
        if cur.child:
            child, tail = cur.child, cur.child
            while tail.next: tail = tail.next
            nxt = cur.next
            cur.next = child
            tail.next = nxt
            cur.child = None
        cur = cur.next
    return head

def wrong_lose_next(head):                  # ghi đè cur.next trước khi lưu
    cur, steps = head, 0
    while cur and steps < 50:               # steps chỉ để demo không treo máy (sai thật sẽ lặp mãi)
        steps += 1
        if cur.child:
            child, tail = cur.child, cur.child
            while tail.next: tail = tail.next
            cur.next = child; child.prev = cur
            tail.next = cur.next            # SAI: lúc này cur.next đã là child
            cur.child = None
        cur = cur.next
    return head

spec = [1, 2, (3, [7, 8]), 4]
for fn in (wrong_keep_child, wrong_no_prev):
    head = build_multilevel(spec)
    ids = {id(n) for n in all_nodes(head)}
    ok, msg = check_flat(fn(head), flat_spec(spec), ids)
    print(fn.__name__, "->", ok, "|", msg)
    assert not ok

head = build_multilevel(spec)
res = wrong_lose_next(head)
print("wrong_lose_next đọc theo next:", read_vals(res)[:12], "(đáng lẽ [1,2,3,7,8,4])")

wrong_keep_child -> False | nút 3 vẫn còn child
wrong_no_prev -> False | prev/next lệch tại nút 3
wrong_lose_next đọc theo next: [1, 2, 3, 7, 8, 7, 8, 7, 8, 7, 8, 7] (đáng lẽ [1,2,3,7,8,4])


### 12.3. Đối chiếu bốn cách trên danh sách đa tầng ngẫu nhiên (seed cố định)

Sinh spec ngẫu nhiên: số nút `1..300`, độ sâu lồng nhau tối đa `7`, giá trị `1..10^5` (có thể trùng). So bốn cách với đáp án `flat_spec`.

In [21]:
def gen_spec(rng, n, depth=0, max_depth=7):
    items, remaining = [], n
    while remaining > 0:
        v = rng.randint(1, 100000)
        remaining -= 1
        if remaining > 0 and depth < max_depth and rng.random() < 0.35:
            sub = rng.randint(1, remaining)
            items.append((v, gen_spec(rng, sub, depth + 1, max_depth)))
            remaining -= sub
        else:
            items.append(v)
    return items

rng = random.Random(430)
cnt = 0
for _ in range(400):
    spec = gen_spec(rng, rng.randint(1, 300), 0, rng.choice([1, 2, 4, 7]))
    run_all(spec)
    cnt += 1
print("đã đối chiếu", cnt, "danh sách đa tầng ngẫu nhiên: OK")

big = gen_spec(rng, 1000, 0, 6)
run_all(big)
print("1000 nút: OK")

đã đối chiếu 400 danh sách đa tầng ngẫu nhiên: OK
1000 nút: OK


### 12.4. Độ sâu lồng nhau 1000: đệ quy hay lặp?

Mỗi nút có `child` kế tiếp tạo chuỗi dài `1000` tầng (đề cho phép đến `1000` nút). Đệ quy sâu 1000 dễ chạm giới hạn đệ quy mặc định của Python.

In [22]:
def build_deep(depth):
    """Chuỗi depth nút, mỗi nút là child của nút trước (dựng bằng vòng lặp)."""
    head = Node(1)
    cur = head
    for v in range(2, depth + 1):
        nxt = Node(v)
        cur.child = nxt
        cur = nxt
    return head

expected = list(range(1, 1001))
for name in ("cach1", "cach3", "cach4", "nop"):
    head = build_deep(1000)
    ids = {id(n) for n in all_nodes(head)}
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(20000) if name == "cach1" else None      # cách 1 dùng đệ quy để gom
    ok, msg = check_flat(ALL[name](head), expected, ids)
    sys.setrecursionlimit(old)
    assert ok, (name, msg)
print("các cách lặp (3, 4) và Cách 1 (đã nâng limit) trên chuỗi con sâu 1000: OK")

try:
    flatten_dfs(build_deep(1000))
    print("Cách 2 qua được ở giới hạn mặc định")
except RecursionError:
    print("Cách 2 (đệ quy): RecursionError ở giới hạn mặc định (như dự đoán)")

old = sys.getrecursionlimit()
sys.setrecursionlimit(10000)
head = build_deep(1000)
ids = {id(n) for n in all_nodes(head)}
assert check_flat(flatten_dfs(head), expected, ids)[0]
sys.setrecursionlimit(old)
print("sau khi nâng recursion limit: OK")

các cách lặp (3, 4) và Cách 1 (đã nâng limit) trên chuỗi con sâu 1000: OK
Cách 2 (đệ quy): RecursionError ở giới hạn mặc định (như dự đoán)
sau khi nâng recursion limit: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên đặt `child = None`

In [23]:
spec = [1, 2, (3, [7, 8]), 4]
head = build_multilevel(spec)
ids = {id(n) for n in all_nodes(head)}
res = wrong_keep_child(head)
print("đọc theo next:", read_vals(res))         # thứ tự đúng...
print(check_flat(res, flat_spec(spec), ids))     # ...nhưng còn child => sai

đọc theo next: [1, 2, 3, 7, 8, 4]
(False, 'nút 3 vẫn còn child')


**Vì sao sai:** đề yêu cầu **mọi `child` phải là `None`**. Chỉ có `next` đúng là chưa đủ — LeetCode kiểm tra cả cấu trúc. **Đúng:** sau khi chèn, `cur.child = None`.

### Lỗi 2: Chỉ nối `next`, quên `prev`

In [24]:
head = build_multilevel(spec)
ids = {id(n) for n in all_nodes(head)}
res = wrong_no_prev(head)
print(check_flat(res, flat_spec(spec), ids))

(False, 'prev/next lệch tại nút 3')


**Vì sao sai:** danh sách **đôi** phải đối xứng: `x.next.prev is x`. Khi chèn có **hai** chỗ nối: `cur ⇄ child` (cần `child.prev = cur`) và `tail ⇄ nxt` (cần `nxt.prev = tail`). **Đúng:** đủ 4 phép gán của mục 3.4.

### Lỗi 3: Làm mất `cur.next` vì ghi đè trước khi lưu

In [25]:
head = build_multilevel(spec)
res = wrong_lose_next(head)
print("đọc theo next:", read_vals(res)[:10], "...")

đọc theo next: [1, 2, 3, 7, 8, 7, 8, 7, 8, 7] ...


**Vì sao sai:** dòng `cur.next = child` ghi đè mất `nxt`, rồi `tail.next = cur.next` lại gán `tail.next = child` (đoạn `child...tail` thành một **vòng kín**, vòng `while cur` thật sự sẽ lặp mãi — ở demo ta chặn bằng biến `steps`). **Đúng:** `nxt = cur.next` **trước tiên**, rồi mới sửa.

### Lỗi 4: Đẩy ngăn xếp sai thứ tự (Cách 3)

In [26]:
def wrong_stack_order(head):
    if head is None:
        return None
    dummy = Node(0); prev = dummy; stack = [head]
    while stack:
        cur = stack.pop()
        prev.next = cur; cur.prev = prev
        if cur.child is not None:
            stack.append(cur.child); cur.child = None     # SAI: đẩy child trước...
        if cur.next is not None:
            stack.append(cur.next)                        # ...next sau => next được pop trước
        prev = cur
    head = dummy.next; head.prev = None
    return head

print("sai :", read_vals(wrong_stack_order(build_multilevel(spec_ex1))))
print("đúng:", flat_spec(spec_ex1))

sai : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
đúng: [1, 2, 3, 7, 8, 11, 12, 9, 10, 4, 5, 6]


**Vì sao sai:** ngăn xếp là LIFO: cái đẩy **sau** được lấy **trước**. Muốn `child` đi trước `next`, phải đẩy `next` **trước**, `child` **sau**.

### Lỗi 5: Quên danh sách rỗng (`head = None`)

In [27]:
def wrong_empty(head):
    dummy = Node(0); prev = dummy; stack = [head]
    while stack:
        cur = stack.pop()
        prev.next = cur                       # cur là None khi head rỗng
        cur.prev = prev
    return dummy.next

try:
    wrong_empty(None)
except AttributeError as e:
    print("AttributeError:", e)
print("đúng:", Solution().flatten(None))

AttributeError: 'NoneType' object has no attribute 'prev' and no __dict__ for setting new attributes
đúng: None


**Vì sao sai:** Ví dụ 3 của đề là `head = []`, tức `None`. **Đúng:** `if head is None: return None` (hoặc vòng `while cur:` tự thoát như Cách 4).

## 14. Mẹo & nhận dạng pattern

- **"Cấu trúc có thêm con trỏ `child` / lồng nhau" ⇒ nghĩ tiền thứ tự (preorder):** thăm nút → thăm `child` → thăm `next`.
- **"Chèn một đoạn vào giữa danh sách" ⇒ nhớ 4 phép gán và LƯU `next` trước.** Mẫu này gặp lại ở *Merge Two Sorted Lists (21)*, *Reverse Linked List II (92)*, *Reorder List (143)*.
- **Duy trì tính hai chiều:** mỗi lần sửa `a.next = b` thì nghĩ ngay "`b.prev = a`?".
- **Tìm cách tránh đệ quy:** nếu có thể "kéo" danh sách con lên tầng chính thì lặp một con trỏ là đủ (Cách 4) — `O(1)` bộ nhớ.
- **Dùng nút giả `dummy`** để khỏi xử lý riêng cho nút đầu.
- Dữ liệu đa tầng: tự viết `build_...` và `show_...` để nhìn được cấu trúc trước/sau; viết **giám khảo** kiểm tra con trỏ hai chiều.

## 15. Biến thể

1. **Flatten Binary Tree to Linked List (114):** làm phẳng cây thành danh sách theo tiền thứ tự (dùng `right` làm `next`) — cùng ý tưởng tiền thứ tự.
2. **Flatten Nested List Iterator (341):** trải phẳng list lồng nhau lười (iterator) bằng ngăn xếp.
3. **Làm phẳng nhưng nối `child` ở **cuối** (sau `next`):** thứ tự khác — cần đi theo từng tầng (BFS).
4. **Unflatten:** đảo ngược — cho danh sách phẳng và thông tin độ sâu, dựng lại `child`.
5. **Copy List with Random Pointer (138):** cấu trúc có con trỏ phụ, cần sao chép đúng; tương tự về việc quản lý nhiều con trỏ.
6. **Danh sách có `child` nhưng đơn (chỉ `next`)**: bỏ `prev`, bài dễ hơn.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Gom nút rồi nối lại | tiền thứ tự vào list, nối `i-1`/`i+1` | `O(n)` | `O(n)` | hiểu đề, oracle |
| 2. DFS trả tail | đệ quy làm phẳng con rồi chèn | `O(n)` | `O(d)` | tư duy đệ quy rõ ràng |
| 3. Ngăn xếp | đẩy `next` rồi `child`; nối theo thứ tự pop | `O(n)` | `O(n)` tệ nhất | tránh đệ quy, dễ hiểu |
| 4. Nối ghép tại chỗ | tìm tail của `child`, chèn giữa `cur` và `nxt` | `O(n)` | **`O(1)`** | **nên nộp** |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `flat_spec([1, (2, [3, 4]), 5])`; (b) `flat_spec([(1, [(2, [3])])])`; (c) với `[1, 2, (3, [4])]` sau khi `Solution().flatten`, `head.next.next.next.val` và `.prev.val` của nút `4`; (d) `Solution().flatten(None)`.

**Bài 2 (tính tay).** Với `spec = [1, (2, [3, (4, [5, 6]), 7]), 8]`: vẽ cấu trúc đa tầng, viết kết quả phẳng và liệt kê **từng lần chèn** (Cách 4): `cur`, danh sách con, `tail`, `nxt`.

**Bài 3 (điền TODO).** Hoàn thành `flatten_todo(head)` theo Cách 4.

In [28]:
def flatten_todo(head):
    cur = head
    while cur is not None:
        # TODO 1: nếu cur.child không None:
        #   TODO 2: child = cur.child; tìm tail = nút cuối của danh sách con (đi next đến hết)
        #   TODO 3: nxt = cur.next  (LƯU TRƯỚC)
        #   TODO 4: cur.next = child; child.prev = cur; tail.next = nxt; nếu nxt: nxt.prev = tail
        #   TODO 5: cur.child = None
        # TODO 6: cur = cur.next
        break
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `max_nesting(head)` trả **độ sâu lồng nhau lớn nhất** (danh sách không có child → `1`, rỗng → `0`) bằng đệ quy; và `count_nodes(head)` đếm tổng số nút kể cả con cháu (không làm phẳng). Kiểm tra: `max_nesting(build_multilevel(spec_ex1)) == 3`, `count_nodes(...) == 12`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `values_preorder(head)` trả list giá trị theo thứ tự làm phẳng **mà không sửa** danh sách (dùng ngăn xếp), kiểm tra `child` vẫn còn nguyên sau khi gọi. (b) **Flatten Binary Tree to Linked List (114):** tự dựng `TreeNode(val, left, right)` và viết `flatten_tree(root)` sửa tại chỗ thành chuỗi chỉ dùng `right` theo tiền thứ tự, `left = None`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1, 2, 3, 4, 5]`; `[1, 2, 3]`; sau flatten `[1,2,3,4]`: nút `4` là nút thứ 4, `4.prev.val = 3`; `None`. Bài 3: nhớ lưu `nxt` **trước** khi sửa.
- **Mức 2 (vừa):** Bài 2: kết quả `[1,2,3,4,5,6,7,8]`; chèn 1: `cur=2`, con `[3,4,7]`, tail `7`, `nxt=8`; chèn 2: `cur=4`, con `[5,6]`, tail `6`, `nxt=7`. Bài 4: `max_nesting = 1 + max(max_nesting(n.child) for n có child, default 0)` duyệt từng nút của tầng.
- **Mức 3 (sâu):** Bài 5(a): ngăn xếp như Cách 3 nhưng chỉ đọc `val`, không gán con trỏ. 5(b): lặp: `if root.left: tail = rightmost(root.left); tail.right = root.right; root.right = root.left; root.left = None`; rồi `root = root.right` — giống hệt "nối ghép" của Cách 4.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert flat_spec([1, (2, [3, 4]), 5]) == [1, 2, 3, 4, 5]
assert flat_spec([(1, [(2, [3])])]) == [1, 2, 3]
h = Solution().flatten(build_multilevel([1, 2, (3, [4])]))
n4 = h.next.next.next
assert n4.val == 4 and n4.prev.val == 3
assert Solution().flatten(None) is None

# Bài 2
spec2 = [1, (2, [3, (4, [5, 6]), 7]), 8]
assert run_all(spec2) == [1, 2, 3, 4, 5, 6, 7, 8]


# Bài 3
def flatten_todo_answer(head):
    cur = head
    while cur is not None:
        if cur.child is not None:
            child = cur.child
            tail = child
            while tail.next is not None:
                tail = tail.next
            nxt = cur.next
            cur.next = child
            child.prev = cur
            tail.next = nxt
            if nxt is not None:
                nxt.prev = tail
            cur.child = None
        cur = cur.next
    return head

for sp in (spec_ex1, spec_ex2, spec_ex3, spec2):
    hd = build_multilevel(sp)
    ids = {id(n) for n in all_nodes(hd)}
    assert check_flat(flatten_todo_answer(hd), flat_spec(sp), ids)[0]


# Bài 4
def max_nesting(head):
    if head is None:
        return 0
    best, cur = 1, head
    while cur is not None:
        if cur.child is not None:
            best = max(best, 1 + max_nesting(cur.child))
        cur = cur.next
    return best

def count_nodes(head):
    total, cur = 0, head
    while cur is not None:
        total += 1 + count_nodes(cur.child)
        cur = cur.next
    return total

assert max_nesting(build_multilevel(spec_ex1)) == 3
assert count_nodes(build_multilevel(spec_ex1)) == 12
assert max_nesting(None) == 0 and max_nesting(build_multilevel([1, 2])) == 1


# Bài 5
def values_preorder(head):
    out = []
    stack = [head] if head else []
    while stack:
        cur = stack.pop()
        out.append(cur.val)
        if cur.next:
            stack.append(cur.next)
        if cur.child:
            stack.append(cur.child)
    return out

hd = build_multilevel(spec_ex1)
assert values_preorder(hd) == flat_spec(spec_ex1)
assert hd.next.next.child is not None and hd.next.next.child.val == 7     # không bị sửa
assert values_preorder(None) == []


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def flatten_tree(root):
    cur = root
    while cur is not None:
        if cur.left is not None:
            tail = cur.left
            while tail.right is not None:
                tail = tail.right
            tail.right = cur.right
            cur.right = cur.left
            cur.left = None
        cur = cur.right
    return root

t = TreeNode(1, TreeNode(2, TreeNode(3), TreeNode(4)), TreeNode(5, None, TreeNode(6)))
flatten_tree(t)
vals, cur = [], t
while cur:
    assert cur.left is None
    vals.append(cur.val); cur = cur.right
assert vals == [1, 2, 3, 4, 5, 6]
print("Đáp án bài tập: OK")

Đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (nối ghép tại chỗ, `O(1)` bộ nhớ phụ)

```python
class Solution:
    def flatten(self, head: 'Optional[Node]') -> 'Optional[Node]':
        cur = head
        while cur:
            if cur.child:
                child = cur.child
                tail = child
                while tail.next:
                    tail = tail.next
                nxt = cur.next            # LƯU TRƯỚC
                cur.next = child
                child.prev = cur
                tail.next = nxt
                if nxt:
                    nxt.prev = tail
                cur.child = None          # nhớ xóa child
            cur = cur.next
        return head
```

**Checklist**

- [ ] Đã **lưu `nxt = cur.next`** trước khi ghi đè `cur.next`?
- [ ] Cập nhật **cả hai chiều** ở **cả hai mối nối**: `child.prev = cur` và `nxt.prev = tail` (có kiểm tra `nxt` khác `None`)?
- [ ] Đặt `cur.child = None` sau khi chèn?
- [ ] Thứ tự: con **trước**, `next` sau (nếu dùng ngăn xếp: đẩy `next` trước, `child` sau)?
- [ ] Xử lý `head is None` (danh sách rỗng) và trường hợp `child` nằm ở nút cuối (`nxt is None`)?
- [ ] `head.prev` của kết quả là `None`; không tạo nút mới?
- [ ] Nếu dùng đệ quy: đã nghĩ tới độ sâu lồng nhau 1000 (RecursionError) — hay đã chọn bản lặp?
- [ ] Không khai báo lại `Node` trong bài nộp?